# 블렌딩 위험지수 (12)

**11번과 같은 데이터만 사용:** `blended_weights`, `local_importance`, `X_shap` (같은 run).

1. 11에서 만든 region별 가중치(blended_weights) + 같은 X_shap  
2. 그리드별로 5그룹 점수(Terrain / 교통량 / 보행인프라 / 속도 / 네트워크) 계산  
3. 비율 40·35·12·8·5 적용 → 위험지수(Risk_Base) 산출 → CSV 저장(노션용)

In [ ]:
# 11번과 동일: 같은 run의 blended_weights, local_importance, X_shap 로드
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path(r"C:\Users\a0109\.jupyter")
BASE = sorted(ROOT.glob("*/data/grf_06_outputs"))[0]
pat = re.compile(r"_(\d{8}_\d{6})\.csv$")

def latest_file(prefix: str) -> Path:
    files = list(BASE.glob(f"{prefix}_*.csv"))
    tagged = [(pat.search(f.name).group(1), f) for f in files if pat.search(f.name)]
    if not tagged:
        raise FileNotFoundError(f"{prefix}_*.csv 없음.")
    tagged.sort(key=lambda x: x[0], reverse=True)
    return tagged[0][1]

p_weights = list(BASE.glob("blended_weights_balanced_*.csv"))
if not p_weights:
    raise FileNotFoundError("blended_weights_balanced_*.csv 없음. 11번 먼저 실행.")
p_weights = sorted(p_weights, key=lambda f: re.search(r"(\d{8}_\d{6})", f.name).group(1) if re.search(r"(\d{8}_\d{6})", f.name) else "", reverse=True)[0]
p_local = latest_file("local_importance")
p_xshap = latest_file("X_shap")

out_df = pd.read_csv(p_weights)
local_df = pd.read_csv(p_local)
X_shap = pd.read_csv(p_xshap)

if len(local_df) != len(X_shap):
    raise ValueError("local과 X_shap 행 수 불일치. 같은 run 파일 사용할 것.")
print("blended_weights:", p_weights.name)
print("local_importance:", p_local.name)
print("X_shap:", p_xshap.name)

blended_weights: blended_weights_balanced_alpha05_20260228_115111.csv
local_importance: local_importance_20260228_115111.csv
X_shap: X_shap_20260228_115111.csv


In [2]:
# 1) region: 11과 동일하게 north_mean 기준 North/South
gid_list = local_df["gid"].values
north_mean = X_shap["north_mean"].values
thr = np.median(north_mean)
region_arr = np.where(north_mean >= thr, "North", "South")

# 2) X_shap 컬럼별 0~1 정규화 후, 11의 blended 가중치로 5그룹 점수 계산
X = X_shap.copy()
for c in X.columns:
    mn, mx = X[c].min(), X[c].max()
    X[c] = (X[c] - mn) / (mx - mn + 1e-12)

groups = ["Terrain", "TrafficVolume", "PedInfra", "TrafficSpeed", "Network"]
by_region_group = {}
for r in ["North", "South"]:
    by_region_group[r] = {g: [] for g in groups}
    sub = out_df[out_df["region"] == r]
    for _, row in sub.iterrows():
        if row["feature"] in X.columns:
            by_region_group[r][row["group"]].append((row["feature"], row["w_region"]))

scores = np.zeros((len(X), 5))
for i in range(len(X)):
    r = region_arr[i]
    for gi, g in enumerate(groups):
        scores[i, gi] = sum(X.iloc[i].get(f, 0) * w for f, w in by_region_group[r][g])

# 3) 그룹별 최댓값으로 나눠 스케일 키움 (상위 격자가 1에 가깝게)
col_max = np.maximum(scores.max(axis=0), 1e-12)
scores_scaled = scores / col_max

# 4) 비율(40/35/12/8/5) 합산 후, 순위 기반 0~1로 매김 → 상위권 격자가 큰 값
TARGET = np.array([0.40, 0.35, 0.12, 0.08, 0.05])
risk_raw = scores_scaled @ TARGET
rank_desc = np.empty(len(risk_raw), dtype=float)
order = np.argsort(-risk_raw)
rank_desc[order] = np.arange(len(risk_raw))
risk_norm = 1.0 - (rank_desc / (len(risk_raw) - 1)) if len(risk_raw) > 1 else np.ones(len(risk_raw))

result = pd.DataFrame({
    "gid": gid_list,
    "region": region_arr,
    "Terrain_final": scores_scaled[:, 0],
    "TrafficVolume_final": scores_scaled[:, 1],
    "PedInfra_final": scores_scaled[:, 2],
    "TrafficSpeed_final": scores_scaled[:, 3],
    "Network_final": scores_scaled[:, 4],
    "Risk_Base": risk_norm,
})
print("Risk_Base (min/mean/max):", round(result["Risk_Base"].min(), 4), round(result["Risk_Base"].mean(), 4), round(result["Risk_Base"].max(), 4))
display(result.head(10))

Risk_Base (min/mean/max): 0.0 0.5 1.0


,gid,region,Terrain_final,TrafficVolume_final,PedInfra_final,TrafficSpeed_final,Network_final,Risk_Base
0,다사681455,North,0.317044,0.000000,0.000000,0.000000,0.000000,0.206478
1,다사681456,North,0.310947,0.000000,0.000000,0.000000,0.000000,0.181460
2,다사682451,South,0.314126,0.000000,0.113822,0.000000,0.000000,0.315893
3,다사682452,South,0.303456,0.000000,0.113822,0.000000,0.000000,0.280702
4,다사682453,South,0.314660,0.180531,0.105966,0.336928,0.285714,0.823316
5,다사682454,South,0.337700,0.000000,0.000000,0.000000,0.000000,0.281013
6,다사682455,South,0.353006,0.072476,0.000000,0.427642,0.142857,0.687532
7,다사682456,South,0.332377,0.000000,0.000000,0.000000,0.000000,0.263158
8,다사682457,South,0.311030,0.000000,0.000000,0.000000,0.000000,0.181979
9,다사682458,South,0.303361,0.000000,0.000000,0.000000,0.000000,0.151043


In [3]:
# 위험지수 CSV 저장 (노션 등 활용)
tag = re.search(r"(\d{8}_\d{6})", p_weights.name).group(1)
out_path = BASE / f"blended_risk_index_{tag}.csv"
result.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장:", out_path)

저장: C:\Users\a0109\.jupyter\1최종_LH\data\grf_06_outputs\blended_risk_index_20260228_115111.csv
